**NOTE** `phimats` environment should be used as kernel

In [1]:
import numpy as np

from PreProcessing import PhysicsConfig, MeshConfig, PreProcessing as PP

from MeshManager import MeshManager

from BoundaryConditions import *

from PostProcessing import WriteXDMF

%load_ext autoreload
%autoreload 2

### Simulation data

In [2]:
SimulName = "CT_115MPa"
# Element sets
nElementSets = 1
# Number of steps to achieve the load
nSteps = 500

### Read mesh file

In [ ]:
# Element name
elementName = "quad"  		# meshio compatible element name
mesh = MeshManager("../CT.msh", elementName)
mesh.WriteMesh("../CT")

In [4]:
# Create the config object
meshConfig = MeshConfig(
    nTotNodes=mesh.get_nTotNodes(),
    nTotElements=mesh.get_nTotElements(),
    nDim=mesh.get_nDim(),
    materialNames=mesh.getMaterialNames(),
)

### Apply mechanics boundary conditions
**NOTE** This is the total load to be achieved in `nSteps`.

In [ ]:
dl = 3e-3 # total displacement [m]

In [ ]:
# Dirichlet BCs list
presBCs = []

# Top pull
presBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("Upper_curve"), dof=1, value=dl))

# Bottom y=0
presBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("Lower_curve"), dof=1, value=0.0))

# Bottom left corner pin (Fixed in X)
presBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("Fix"), dof=0, value=0.0))

# Write vtu for visualization 
WriteBCVTK(SimulName+"_mech", mesh, presBCs, dofNames=['ux', 'uy'])

### Apply diffusion boundary conditions

In [9]:
PH2 =  115e6  # Pa
S_sieverts = 6.2e-6 # mol/(m3.Pa^0.5)

S0 = 0.2024
S_sieverts = S0*np.exp(-24.7e3/(8.31*300))

cL = S_sieverts*np.sqrt(PH2)

In [ ]:
# Dirichlet BCs list
conBCs = []
exitNods = []

surface_nodes = np.concatenate([mesh.getNodesByGroup("Surface"), mesh.getNodesByGroup("Upper_curve"), mesh.getNodesByGroup("Lower_curve")])

# Surface con
conBCs.extend(AssignDirichletBC(surface_nodes, dof=0, value=cL))

# Write external 
WriteBCVTK(SimulName+"_diff", mesh, conBCs, dofNames=['con'])

In [11]:
conBCs = [[row[0], row[-1]] for row in conBCs]

### Mechanical material data

In [13]:
# Analysis type
AnalysisType = "PlaneStrainPFF"
# Isotropy
Isotropy = "Isotropic"
# Young's modulus
Emod = 210e9
# Poisson's ratio
nu = 0.3

# Plasticity type
Plasticity = "IsoHard"
HardeningLaw = "PowerLaw"
sig_y0 = 360e6    # Yield strength (Pa)
K_hard = 550e6    # Pa
n_pow = 0.1

rho_0 = 1e11  # Initial dislocation density (m⁻²)
M = 3 		  # Taylor factor
alpha = 0.3   # Dislocation interaction constant
b = 2.5e-10   # Burgers vector (m)
G = Emod/(2*(1+nu)) # Shear modulus
k1 = 2*(G/(200*G*alpha*b))  # Multiplication coefficient
k2 = 10  # Recovery coefficient

MechMaterial = {
	"CT" : {
		"Elastic" : {
			"AnalysisType" : AnalysisType,
			"Isotropy" 	 : Isotropy,
			"Emod" 		 : Emod,
			"nu"   		 : nu,
		},
       	"Plastic" : {
			"Plasticity"   : Plasticity,
			"HardeningLaw" : HardeningLaw,
			"sig_y0"       : sig_y0,
			"K_hard"       : K_hard,
			"n_pow"        : n_pow,
			"rho_0"        : rho_0,
			"M"            : M,
			"b"            : b,
			"alpha"        : alpha,
			"k1"           : k1,
   			"k2"           : k2,
		},
	},
}

### Diffusion material data

In [ ]:
# Trapping --------------

Erho = 20000  # Hydroge dislocation enthalpy (positive) [J/mol]

Vm = 7.09e-6  # Molar volume of Fe [m³/mol]
Vrho = 2*Vm   # Molar volume around dislocation  [m³/mol]
Vh = 2e-6     # Partial molar volume of hydrogen in Fe [m³]
N = 6          
R = 8.31      # Universal gas constant [J/(molK)]
T = 300       # Temperature [K]

theta_b = cL*Vm/N
print(theta_b)

Krho   = np.exp(Erho/(R*T))
theta_rho = theta_b*Krho 

Vrho = 2*Vm
c_rho = theta_rho/Vrho
print("c_rho", c_rho)
Z_rho = c_rho/cL
print("Z_rho", Z_rho)
zeta_rho = R*T*np.log(Z_rho)
print("zeta_rho", zeta_rho)

In [ ]:
# Diffusivity --------------
v = 2e-6
dt = dl/v/nSteps   # Time increment [s]
s = 1      # Trapping capacity 
m = 0      # Dislocation diffusivity ratio

D0x = 8.45e-8; DQx = 5000  # Diffusivity m^2/s

DL = D0x*np.exp(-DQx/(R*T))

Zd = 10

print(DL)

In [ ]:
DiffMaterial = {
    "CT" : {
		"D0x" : D0x,
		"D0y" : D0x,
		"DQx" : DQx,
		"DQy" : DQx,
		"m"   : m,
		"s"   : s,
		"Vh"  : Vh,
		"zeta_rho"   : zeta_rho,
		"Zd"  : Zd,
    }
}

DiffMaterial

### PFF material data

In [ ]:
import matplotlib.pyplot as plt

wc0 = 60e6           # Critical work density [J/m³]
wc_min = wc0*0.15

c_hat = np.linspace(0,10)
c_DTB = cL*3
c_crit = 1.08

print("C_crit: ", c_crit)

beta = 2
wc1 = wc_min + (wc0 - wc_min) * np.exp(-beta * c_hat)

fig, ax = plt.subplots(figsize=(2*56.25/25.4, 1.5*56.25/25.4), dpi=100, tight_layout=True)

ax.plot(c_hat, wc1, color="r") 

ax.set_xlabel("$\\bar{c} =  c/c_\\mathrm{crit}$")
ax.set_ylabel("$\\tilde{w}_\\mathrm{c}(\\bar{c}$) (J/m³)")

In [ ]:
const_ell = 3e-5      # Length-scale regularization parameter [m]
Gc = 2*const_ell*wc0  # Fracture toughness [J/m²]

PFFMaterial = {
	"CT" : {
		"ChemoMech" : {
			"const_ell" : const_ell,
			"wc" 	    : wc0,
   			"wc_min" 	: wc_min, 
   			"beta" 	    : beta,
      		"c_DTB"     : c_DTB,
      		"c_crit" 	: c_crit,
		}
    }
}

print("l: ", const_ell, " m")
print("Gc: ", Gc, " J/m²")
print("w_c0: ", wc0, " J/m³")
print("w_cmin: ", wc_min, " J/m³")

### Initialize simulation object

### Mechanical

In [21]:
# Create the config object
mechPhysConfig = PhysicsConfig(
    SimulName = SimulName,
    PhysicsType = "Mechanical",
    PhysicsCategory = "Plastic",
    nSteps    = nSteps,
    presBCs=presBCs
)

In [ ]:
MechNotch = PP(mechPhysConfig, meshConfig, MechMaterial)

In [ ]:
MechNotch.WriteInputFile()

### Diffusion

In [24]:
# Create the config object
diffPhysConfig = PhysicsConfig(
    SimulName = SimulName,
    PhysicsType = "Transport",
    PhysicsCategory = "MechTrappingPFF",
    nSteps    = nSteps,
    dt = dt,
    conB=cL,
    presBCs=conBCs,
)

In [ ]:
DiffNotch = PP(diffPhysConfig, meshConfig, DiffMaterial)

In [ ]:
DiffNotch.WriteInputFile()

#### PFF

In [27]:
# Create the config object
pffPhysConfig = PhysicsConfig(
    SimulName = SimulName,
    PhysicsType = "PFF",
    PhysicsCategory = list(PFFMaterial["CT"].keys())[0],
    nSteps    = nSteps,
)

In [ ]:
PFFNotch = PP(pffPhysConfig, meshConfig, PFFMaterial)

In [ ]:
PFFNotch.WriteInputFile()

### Write output files

In [ ]:
OVERWRITE = False

In [ ]:
MechNotch.WriteOutputFile(overwrite=OVERWRITE)
DiffNotch.WriteOutputFile(OVERWRITE=True, AVCON=True, FLUX=True)
PFFNotch.WriteOutputFile(overwrite=OVERWRITE)

In [ ]:

WriteXDMF(SimulName, # Base simulation name
          "../CT",   # Mesh file name
          "quad",    # Element name
          nSteps+1,  # Number of steps 
          components=["mech", "pff", "diff"],
    nDim=2,
    mechModel="Plastic", # Adds strain_p, stress_eq, etc.
    FLUX=True)

**PETSc command line options**

```
./CT_Hydrogen -snes_linesearch_type cp -snes_linesearch_damping 0.5 -snes_linesearch_monitor -snes_linesearch_max_it 50 -snes_max_it 500
```